### MMR (Maximal Marginal Relevence) :: Imp


uses for reduce redundancy in the retrieved results while maintaining high relevence to the query.

In [1]:
from langchain_core.documents import Document

In [2]:
# Sample Data : 
documents = [
    Document(
        page_content="Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.",
        metadata={"source": "ml_notes"}
    ),

    Document(
        page_content="Deep Learning uses neural networks with multiple layers to solve complex problems such as image recognition and speech processing.",
        metadata={"source": "dl_notes"}
    ),

    Document(
        page_content="Natural Language Processing enables computers to understand, interpret, and generate human language.",
        metadata={"source": "nlp_notes"}
    ),

    Document(
        page_content="Retrieval Augmented Generation combines vector search and large language models to provide accurate answers from external knowledge sources.",
        metadata={"source": "rag_notes"}
    ),

    Document(
        page_content="Vector databases such as Chroma and FAISS store embeddings and perform semantic similarity search.",
        metadata={"source": "vector_db_notes"}
    )
]

In [3]:
%pip install faiss-cpu


Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Step 1  : Initialize GenAi Embeddings

embedding_model = GoogleGenerativeAIEmbeddings(
     model="gemini-embedding-001"
)

# Step 2  :  Create the FAISS vector store from documents

vectorStore = FAISS.from_documents(
    documents = documents,
    embedding = embedding_model
)

/tmp/ipykernel_6852/2730526668.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/media/ayush-paliwal/New Volume2/LangChain Models/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
# Enable MMR in the retriever :

retriever = vectorStore.as_retriever(
    search_type = "mmr",   # <----------- This enables MMR 
    search_kwargs = {"k" : 3 , "lambda_mult" : 0.4} # k = top results , lambda_mult = relevance_diversity balance 
)

In [6]:
query = "What is ML ?"
result = retriever.invoke(query)

In [7]:
for i, docs in enumerate(result):
    print(f"\n--- Result {i+1} ----")
    print(docs.page_content)
    


--- Result 1 ----
Machine Learning is a subset of Artificial Intelligence that enables systems to learn from data.

--- Result 2 ----
Vector databases such as Chroma and FAISS store embeddings and perform semantic similarity search.

--- Result 3 ----
Retrieval Augmented Generation combines vector search and large language models to provide accurate answers from external knowledge sources.


## Multi-Query Retriever :: Imp

In [ ]:
%pip install -U langchain langchain-core langchain-community

Note: you may need to restart the kernel to use updated packages.


In [29]:
%pip install langchain-classic


Note: you may need to restart the kernel to use updated packages.


In [11]:
import langchain
print(langchain.__version__)

1.3.4


In [23]:
import pkgutil

print("langchain.retrievers" in [m.name for m in pkgutil.iter_modules()])

False


In [30]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import MultiQueryRetriever



In [31]:
documents = [
    Document(
        page_content="""
        Machine Learning is a field of Artificial Intelligence that allows
        computers to learn patterns from data and make predictions.
        """
    ),

    Document(
        page_content="""
        Deep Learning is a subset of Machine Learning that uses neural
        networks with multiple hidden layers.
        """
    ),

    Document(
        page_content="""
        Neural Networks are computational models inspired by the human brain.
        They are widely used in Deep Learning applications.
        """
    ),

    Document(
        page_content="""
        Natural Language Processing enables computers to understand,
        analyze, and generate human language.
        """
    ),

    Document(
        page_content="""
        Large Language Models such as Gemini and GPT are trained on
        massive text datasets and are capable of generating human-like text.
        """
    ),

    Document(
        page_content="""
        Retrieval-Augmented Generation combines information retrieval
        with language models to provide more accurate answers.
        """
    ),

    Document(
        page_content="""
        Vector databases store embeddings and perform semantic search.
        Examples include Chroma, FAISS, Pinecone, and Qdrant.
        """
    ),

    Document(
        page_content="""
        Embeddings are numerical vector representations of text that capture
        semantic meaning and relationships between words.
        """
    )
]

In [32]:
import pkgutil
import langchain

for mod in pkgutil.walk_packages(langchain.__path__, prefix="langchain."):
    if "multi" in mod.name.lower():
        print(mod.name)

In [33]:
# Step 1  : Initialize GenAi Embeddings

embedding_model = GoogleGenerativeAIEmbeddings(
     model="gemini-embedding-001"
)


In [34]:
# Step 2  :  Create the FAISS vector store from documents

vectorStore2 = FAISS.from_documents(
    documents = documents,
    embedding = embedding_model
)

In [35]:
#  create retriever : 

similarity_retriever = vectorStore2.as_retriever(search_type = "similarity" , search_kwargs = {"k" : 5})

In [36]:
# multiquery retriever : 

multiqueryRetriever = MultiQueryRetriever.from_llm(
     retriever = vectorStore2.as_retriever(search_kwargs = {"k" : 5}),
     llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
)


In [37]:
# Query : 

query = "What is best to use ?"

In [38]:
# Retriever results : 

similarity_result = similarity_retriever.invoke(query)
multiquery_result = multiqueryRetriever.invoke(query)


In [40]:
for i,doc in enumerate(similarity_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("+" * 150)

for i,doc in enumerate(multiquery_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---

        Vector databases store embeddings and perform semantic search.
        Examples include Chroma, FAISS, Pinecone, and Qdrant.
        

--- Result 2 ---

        Retrieval-Augmented Generation combines information retrieval
        with language models to provide more accurate answers.
        

--- Result 3 ---

        Deep Learning is a subset of Machine Learning that uses neural
        networks with multiple hidden layers.
        

--- Result 4 ---

        Embeddings are numerical vector representations of text that capture
        semantic meaning and relationships between words.
        

--- Result 5 ---

        Neural Networks are computational models inspired by the human brain.
        They are widely used in Deep Learning applications.
        
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

--- Result 1 ---

        Natural Language Processin

### Contextual Comperssion Retriever :: Imp

In [41]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [43]:
documents2 = [

Document(page_content="""
Machine Learning is a field of Artificial Intelligence that allows computers to learn patterns from data and make predictions.
In simple terms, ML helps systems improve automatically with experience.
Machine Learning models can be trained using supervised or unsupervised learning techniques.
It is widely used in recommendation systems, spam detection, and fraud detection.
"""),

Document(page_content="""
Deep Learning is a subset of Machine Learning that uses neural networks with multiple hidden layers.
These deep neural networks can automatically extract features from raw data.
Deep Learning is powerful but requires large datasets and high computation power.
It is commonly used in image recognition, speech processing, and natural language understanding.
"""),

Document(page_content="""
Neural Networks are computing systems inspired by the human brain.
They consist of layers of nodes including input layer, hidden layers, and output layer.
Each connection has weights that are adjusted during training.
Neural networks are the foundation of deep learning models.
"""),

Document(page_content="""
Natural Language Processing (NLP) helps computers understand and generate human language.
NLP includes tasks like sentiment analysis, translation, and question answering.
Even though NLP is part of AI, it heavily depends on machine learning and deep learning.
Real-world NLP systems often use transformers and large language models.
"""),

Document(page_content="""
Large Language Models like GPT and Gemini are trained on massive text datasets.
They predict the next word in a sequence to generate human-like text.
These models can perform tasks like summarization, coding, and reasoning.
However, they may sometimes hallucinate incorrect information.
"""),

Document(page_content="""
Retrieval-Augmented Generation (RAG) combines search with generative AI models.
Instead of relying only on memory, RAG retrieves relevant documents from a vector database.
Then it uses a language model to generate an answer based on retrieved context.
RAG improves factual accuracy in AI systems.
"""),

Document(page_content="""
Vector databases store embeddings which are numerical representations of text.
They allow semantic search instead of keyword search.
Examples include FAISS, Chroma, Pinecone, and Qdrant.
These databases are essential for building RAG systems.
"""),

Document(page_content="""
Embeddings convert text into high-dimensional vectors.
Texts with similar meaning have similar vector representations.
This enables semantic search and similarity matching in AI applications.
Embedding models are usually trained using large-scale language data.
""")
]

In [49]:
vectorStore3 = FAISS.from_documents(documents2 , embedding_model)

In [50]:
base_retriever = vectorStore3.as_retriever(search_kwargs={"k" : 3})

In [51]:
# Set up the compressor using an LLM

llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
compressor = LLMChainExtractor.from_llm(llm)

In [53]:
# Create the contextual compressor retriever

compressor_retriever = ContextualCompressionRetriever(
     base_retriever= base_retriever,
     base_compressor= compressor
)

In [57]:
# Query the retriever :

query = " What is Ai ?"
compressed_results = compressor_retriever.invoke(query)

In [58]:
for i,doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Even though NLP is part of AI
